## Import standart libraries

In [1]:
import numpy as np
import pandas as pd

## Import sys library 
We import this library so we can add new path, and then use our Py_MSSql module

In [2]:
import sys
sys.path.append('Modules/')

In [3]:
from Py_MSSql import Py_MSSql

## Define the connection to MSSQL

In [4]:
server = "localhost\\SQLEXPRESS;Trusted_Connection=True;"#" Integrated Security=True;"
database = 'Test'

py_mssql = Py_MSSql(server=server,database=database)

## Create data for the example

In [5]:
df = pd.DataFrame(np.random.randint (1,11,(1000000,6)), 
                   columns=['a','b','c','d','e','f'])

In [6]:
df2 = pd.DataFrame(np.random.randint (1,11,(1000000,6)), 
                   columns=['a','b','c','d','e','f'])

In [7]:
df3 = pd.DataFrame(np.random.randint (1,11,(1000000,6)), 
                   columns=['a','b','c','d','e','f'])

In [8]:
import os
if not os.path.exists('my_data'):
    os.makedirs('my_data')

In [9]:
df.to_csv('my_data/df1.csv', index=False)

In [10]:
df2.to_csv('my_data/df2.csv', index=False)

In [11]:
df3.to_csv('my_data/df3.csv', index=False)

## Option 1 - merge all csv files into one df, and then insert into mssql table

In [12]:
# Using os library
# get a list of all data files in the directory
path = "my_data/"

list_of_files = os.listdir(path)

print(list_of_files)


['df1.csv', 'df2.csv', 'df3.csv']


In [18]:
# Import the first data file in the list, As Pandas DF

stock_file = list_of_files[0]

print(stock_file)
print(type(stock_file))

df = pd.read_csv(path + stock_file)

df1.csv
<class 'str'>


In [19]:
# Append the rest of the data files to the pandas DF, Using a loop
# I assume that we already created df using index 0

for stock_file in list_of_files[1:]:
    
    temp_df = pd.read_csv(path + stock_file)
    
    df = df.append(temp_df)
    
    del temp_df
    print ('done '+ stock_file)

done df2.csv
done df3.csv


In [20]:
tbl_name = 'df_table'

In [21]:
py_mssql.drop_table_if_exist(tbl_name)

IF OBJECT_ID('dbo.df_table', 'U') IS NOT NULL DROP TABLE dbo.df_table


In [22]:
py_mssql.create_tbl_from_df(df,tbl_name)

CREATE TABLE df_table (a NUMERIC(18,10), b NUMERIC(18,10), c NUMERIC(18,10), d NUMERIC(18,10), e NUMERIC(18,10), f NUMERIC(18,10))


In [23]:
py_mssql.insert_into_tbl_from_df(df,tbl_name) 

?,?,?,?,?,?, 16
INSERT QUERY:INSERT INTO df_table (a, b, c, d, e, f) VALUES(?,?,?,?,?,?)


True

## Option 2 -  Insert each csv file into mssql table

In [25]:
# Using os library
# get a list of all data files in the directory
path = "my_data/"

list_of_files = os.listdir(path)

print(list_of_files)


['df1.csv', 'df2.csv', 'df3.csv']


In [32]:
# loop on all files, and create them in mssql
i = 1

for stock_file in list_of_files:
    
    # Read csv
    temp_df = pd.read_csv(path + stock_file)
    
    #insert into mssql table
    tbl_name = 'A' + str (i)
    py_mssql.drop_table_if_exist(tbl_name)
    py_mssql.create_tbl_from_df(df,tbl_name)
    py_mssql.insert_into_tbl_from_df(df,tbl_name) 
    
    del temp_df
    print ('\ndone '+ stock_file)
    i += 1

IF OBJECT_ID('dbo.A1', 'U') IS NOT NULL DROP TABLE dbo.A1
CREATE TABLE A1 (a NUMERIC(18,10), b NUMERIC(18,10), c NUMERIC(18,10), d NUMERIC(18,10), e NUMERIC(18,10), f NUMERIC(18,10))
?,?,?,?,?,?, 16
INSERT QUERY:INSERT INTO A1 (a, b, c, d, e, f) VALUES(?,?,?,?,?,?)

done df1.csv
IF OBJECT_ID('dbo.A2', 'U') IS NOT NULL DROP TABLE dbo.A2
CREATE TABLE A2 (a NUMERIC(18,10), b NUMERIC(18,10), c NUMERIC(18,10), d NUMERIC(18,10), e NUMERIC(18,10), f NUMERIC(18,10))
?,?,?,?,?,?, 16
INSERT QUERY:INSERT INTO A2 (a, b, c, d, e, f) VALUES(?,?,?,?,?,?)

done df2.csv
IF OBJECT_ID('dbo.A3', 'U') IS NOT NULL DROP TABLE dbo.A3
CREATE TABLE A3 (a NUMERIC(18,10), b NUMERIC(18,10), c NUMERIC(18,10), d NUMERIC(18,10), e NUMERIC(18,10), f NUMERIC(18,10))
?,?,?,?,?,?, 16
INSERT QUERY:INSERT INTO A3 (a, b, c, d, e, f) VALUES(?,?,?,?,?,?)

done df3.csv
